# Oscillating (Orbiting) Charge

Rebuilt from `legacy/Public CEM(Not My Code).py` -- that script leaned on
the third-party `pycharge` library for the retarded-field computation and
carried no license/attribution despite its own filename flagging it as
not the user's code. This notebook does the same physical demo (a charge
on a circular path, animated field-magnitude heatmap + direction quiver)
using `retarded_fields.py`, validated in `Validation.ipynb` against
Coulomb's law, the boosted-Coulomb closed form, and the Larmor formula --
not borrowed, and checked rather than just visually plausible.

In [1]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib
from matplotlib.colors import LogNorm
from numba import njit
import imageio_ffmpeg
matplotlib.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

import retarded_fields as rf

os.makedirs('media', exist_ok=True)
c, eps0, q = 1.0, 1.0, 1.0

In [2]:
r_orbit = 1.0
Omega = 0.5  # v = r_orbit*Omega = 0.5c -- mildly relativistic, for a visibly non-trivial radiation pattern
T_period = 2 * np.pi / Omega
print(f'orbit speed = {r_orbit * Omega}c, period = {T_period:.3f}')

# njit path functions (returning plain (x, y, z) tuples, not np.array) --
# required by rf.lw_fields_grid_jit, the numba-accelerated grid solver used
# below for the animation. See the comment above _retarded_time_jit in
# retarded_fields.py for why: JIT-compiling this per-grid-point,
# per-Newton-iteration inner loop turns a ~200ms/frame computation into
# ~0.2ms/frame.
@njit(cache=True)
def r_func(t):
    return r_orbit * math.cos(Omega * t), r_orbit * math.sin(Omega * t), 0.0

@njit(cache=True)
def v_func(t):
    return -r_orbit * Omega * math.sin(Omega * t), r_orbit * Omega * math.cos(Omega * t), 0.0

@njit(cache=True)
def a_func(t):
    x, y, _ = r_func(t)
    return -Omega**2 * x, -Omega**2 * y, 0.0

N = 70
extent = 6.0
xs = np.linspace(-extent, extent, N)
X, Y = np.meshgrid(xs, xs, indexing='ij')
Z = np.zeros_like(X)

orbit speed = 0.5c, period = 12.566


In [3]:
n_frames = 60
times = np.linspace(0, T_period, n_frames, endpoint=False)
E_frames = np.zeros((n_frames, N, N, 3))
t_ret_guess = None
for i, t in enumerate(times):
    E_frames[i], _, t_ret_guess = rf.lw_fields_grid_jit(r_func, v_func, a_func, X, Y, Z, t=t, q=q, c=c, eps0=eps0,
                                                          t_ret_guess=t_ret_guess)
    print(f'frame {i+1}/{n_frames}', end='\r')
print()
E_mag = np.linalg.norm(E_frames, axis=-1)
print('field magnitude range:', E_mag.min(), E_mag.max())

frame 60/60
field magnitude range: 0.0011128215468882293 53.96271643548907


In [4]:
fig, ax = plt.subplots(figsize=(6, 6))
vmin, vmax = np.percentile(E_mag[E_mag > 0], [5, 99.5])
im = ax.imshow(E_mag[0].T, origin='lower', extent=(-extent, extent, -extent, extent),
               cmap='inferno', norm=LogNorm(vmin=vmin, vmax=vmax))
quiver_stride = 6
qx, qy = X[::quiver_stride, ::quiver_stride], Y[::quiver_stride, ::quiver_stride]

def unit_components(i):
    Ex_i = E_frames[i, ::quiver_stride, ::quiver_stride, 0]
    Ey_i = E_frames[i, ::quiver_stride, ::quiver_stride, 1]
    mag = np.hypot(Ex_i, Ey_i)
    mag[mag == 0] = 1
    return Ex_i / mag, Ey_i / mag

# Fixed numeric scale (never scale=None/autoscale) so every frame's
# unit-length direction arrows draw at the same on-screen size instead of
# rescaling frame-to-frame against whatever raw magnitude range (spanning
# 5 orders of magnitude near vs. far from the charge) happened to be
# passed in at quiver-creation time.
grid_spacing = (2 * extent) / (N - 1)
cell_spacing = quiver_stride * grid_spacing
arrow_scale = 1.0 / (0.8 * cell_spacing)

Ux0, Uy0 = unit_components(0)
quiv = ax.quiver(qx, qy, Ux0, Uy0, color='cyan', scale=arrow_scale, scale_units='xy',
                  pivot='mid', alpha=0.6)
charge_dot, = ax.plot([], [], 'wo', markersize=6)
ax.set_title('Radiated field of an orbiting charge (v=0.5c)')

def animate(i):
    im.set_array(E_mag[i].T)
    Ux_i, Uy_i = unit_components(i)
    quiv.set_UVC(Ux_i, Uy_i)
    pos = r_func(times[i])
    charge_dot.set_data([pos[0]], [pos[1]])
    return im, quiv, charge_dot

# blit=False: with blit=True the quiver's background wasn't being restored
# correctly between frames when saving to file, leaving every prior
# frame's arrows visible underneath the current one (the actual cause of
# the "messy" look, on top of the missing fixed scale above).
ani = animation.FuncAnimation(fig, animate, frames=n_frames, interval=60, blit=False)
ani.save('media/oscillating_charge.mp4', writer='ffmpeg', fps=20, dpi=130)
plt.close(fig)
print('saved media/oscillating_charge.mp4')

saved media/oscillating_charge.mp4
